# NeuralAtmosphereOperator — từ FNO đến SFNO và thiết kế model

> **Phạm vi của notebook.** Tài liệu này chỉ mô tả **model**: bài toán dưới góc nhìn neural operator, nền tảng FNO, lý do chuyển sang SFNO, kiến trúc được triển khai, cấu hình, số tham số và hàm loss. Notebook **không** trình bày training pipeline, cách chia dữ liệu, DataLoader, data normalization, optimizer, scheduler, checkpoint, evaluation hay deployment.

Model được mô tả là baseline hiện tại của repository: **SFNO-SC2-L6-E128**, nhận một trạng thái khí quyển toàn cầu 26 kênh trên lưới $361\times720$ và dự báo trạng thái kế tiếp. Đây là một biến thể nhỏ gọn thuộc cùng họ SFNO, không phải bản sao nguyên xi của cấu hình lớn trong Makani.

> **Quy ước thuật ngữ.** Notebook giữ nguyên những thuật ngữ tiếng Anh đã phổ biến trong machine learning và scientific computing—chẳng hạn *embedding*, *spectral convolution*, *mode truncation*, *residual connection*, *rollout* và *loss*—khi bản dịch tiếng Việt làm sai sắc thái kỹ thuật hoặc khiến câu văn thiếu tự nhiên. Thuật ngữ được giải thích tại lần xuất hiện đầu tiên.

Nguồn chính: [Li et al., Fourier Neural Operator, ICLR 2021](https://arxiv.org/abs/2010.08895), [Bonev et al., Spherical Fourier Neural Operators, ICML 2023](https://proceedings.mlr.press/v202/bonev23a.html), và phiên bản triển khai đã khóa [NVIDIA torch-harmonics v0.7.4](https://github.com/NVIDIA/torch-harmonics/tree/v0.7.4).

## 1. Lộ trình lập luận

Tài liệu đi theo thứ tự sau:

1. Biểu diễn dự báo khí quyển như một ánh xạ giữa các **trường vật lý**.
2. Từ neural operator tổng quát đến **Fourier Neural Operator (FNO)** trên miền phẳng, tuần hoàn.
3. Chỉ ra vì sao FFT hai chiều không phù hợp hoàn toàn với hình học Trái Đất.
4. Thay planar Fourier basis bằng **spherical harmonics** để thu được SFNO.
5. Bóc tách kiến trúc SFNO cụ thể của dự án, từ tensor đầu vào đến đầu ra.
6. Giải thích từng lựa chọn cấu hình, số mode và số tham số.
7. Xây dựng hàm loss tương thích với diện tích mặt cầu và nhiều đại lượng vật lý.

Thứ tự này quan trọng: SFNO không phải một mạng hoàn toàn tách biệt với FNO; nó thay planar Fourier transform trong FNO bằng một spectral transform phù hợp với miền $\mathbb S^2$.

## 2. Bài toán dự báo dưới góc nhìn toán tử

Gọi trạng thái khí quyển tại thời điểm $t$ là trường nhiều kênh

$$
\mathbf{x}_t : \mathbb S^2 \rightarrow \mathbb R^{C}, \qquad C=26.
$$

Tại mỗi vị trí trên bề mặt cầu, $\mathbf{x}_t$ chứa 26 đại lượng bề mặt hoặc theo mực áp suất. Bài toán one-step forecasting là xấp xỉ toán tử tiến hóa

$$
\mathcal G_{\Delta t}: \mathbf{x}_t \mapsto \mathbf{x}_{t+\Delta t}.
$$

Một neural network thông thường xử lý tensor ảnh có thể được xem như ánh xạ giữa hai không gian hữu hạn chiều. Neural operator nhấn mạnh rằng đối tượng cần học về bản chất là ánh xạ giữa các **function spaces**:

$$
\mathcal G_\theta : \mathcal A \subseteq \{\mathbb S^2\to\mathbb R^{C_{in}}\}
\longrightarrow
\mathcal U \subseteq \{\mathbb S^2\to\mathbb R^{C_{out}}\}.
$$

Lưới $361\times720$ chỉ là một phép rời rạc hóa của các trường đó. Góc nhìn toán tử phù hợp với động lực học khí quyển vì cùng một quy luật tiến hóa phải tác động lên toàn bộ các trạng thái, thay vì ghi nhớ từng bản đồ riêng lẻ. Tuy vậy, một model học từ dữ liệu vẫn chỉ xấp xỉ toán tử trên phân bố và độ phân giải đã quan sát; không nên diễn giải đây là bộ giải PDE chính xác hoặc đảm bảo tổng quát hóa ở mọi lưới.

## 3. Neural operator tổng quát

Một lớp neural operator có thể được viết dưới dạng

$$
v_{l+1}(x)=\sigma\!\left(W_l v_l(x)+\left(\mathcal K_l v_l\right)(x)\right),
$$

với toán tử tích phân

$$
(\mathcal K_l v)(x)=\int_D \kappa_l(x,y) v(y)\,dy.
$$

Trong đó:

- $v_l(x)$ là hidden representation tại vị trí $x$;
- $W_l$ là biến đổi cục bộ theo kênh;
- $\kappa_l(x,y)$ là kernel học được, cho phép vị trí $x$ nhận thông tin từ vị trí $y$;
- $\sigma$ là nonlinear activation, giúp nhiều lớp liên tiếp mô tả được động lực học phức tạp.

Nếu tính tích phân trực tiếp trên mọi cặp $(x,y)$, chi phí tăng gần bậc hai theo số điểm lưới. Ý tưởng trung tâm của FNO là tham số hóa kernel và thực hiện global mixing trong miền Fourier, nơi convolution trở thành phép nhân.

# Phần I — Fourier Neural Operator (FNO)

## 4. Từ convolution đến spectral convolution

Trên miền phẳng tuần hoàn $\mathbb T^2=S^1\times S^1$, biến đổi Fourier của trường $v$ là

$$
\widehat v(k_1,k_2)=\int_{\mathbb T^2}v(x_1,x_2)
e^{-2\pi i(k_1x_1+k_2x_2)}\,dx_1dx_2.
$$

Convolution theorem cho phép viết

$$
\mathcal F[\kappa * v](k)=\widehat\kappa(k)\,\widehat v(k).
$$

FNO thay $\widehat\kappa(k)$ bằng tensor phức học được $R_\theta(k)$ và định nghĩa spectral convolution

$$
\mathcal K_\theta(v)=\mathcal F^{-1}\!\left(R_\theta\,\mathcal F(v)\right).
$$

Với nhiều kênh, tại mỗi mode $k$ ta thực hiện channel mixing

$$
\widehat z_o(k)=\sum_{c=1}^{C_{in}}R_{o,c}(k)\widehat v_c(k).
$$

Do đó một mode đầu ra có thể kết hợp thông tin từ mọi kênh đầu vào. Vì mỗi hệ số Fourier được tính từ toàn miền, một lớp spectral đã có receptive field toàn cục. Đây là ưu điểm quan trọng đối với các hệ có tương tác xa như hoàn lưu khí quyển.

## 5. Cấu trúc chuẩn của một FNO

Một FNO điển hình gồm ba giai đoạn:

### 5.1 Lifting

Ánh xạ số kênh vật lý $C_{in}$ sang embedding dimension $E$:

$$v_0(x)=P\mathbf{x}(x), \qquad P\in\mathbb R^{E\times C_{in}}.$$

Trong CNN, đây tương đương convolution $1\times1$: chỉ thực hiện channel mixing tại cùng vị trí, chưa thực hiện spatial mixing.

### 5.2 Các Fourier block

Dạng khái niệm thường gặp là

$$v_{l+1}=\sigma\left(W_l v_l+\mathcal K_{\theta_l}v_l\right).$$

Nhánh spectral đảm nhiệm tương tác toàn cục; nhánh pointwise/skip giữ thông tin cục bộ và tạo đường truyền gradient. Các implementation hiện đại có thể đặt normalization, MLP và residual theo block topology khác nhau, nhưng nguyên lý spectral vẫn không đổi.

### 5.3 Projection

Ánh xạ embedding trở lại $C_{out}$ kênh dự báo:

$$\widehat{\mathbf y}(x)=Qv_L(x), \qquad Q\in\mathbb R^{C_{out}\times E}.$$

## 6. Mode truncation: vì sao FNO không học toàn bộ phổ

FNO thường chỉ giữ tập mode tần số thấp $\Lambda$:

$$
\widehat z(k)=
\begin{cases}
R_\theta(k)\widehat v(k), & k\in\Lambda,\\
0, & k\notin\Lambda.
\end{cases}
$$

Spectral truncation có ba tác dụng:

- giảm số tham số và chi phí;
- tập trung capacity vào cấu trúc quy mô lớn, thường chứa phần lớn năng lượng của trường trơn;
- tạo một bottleneck phổ có tác dụng regularization.

Mode truncation cũng đặt ra giới hạn biểu diễn: chi tiết nhỏ hơn bước sóng tương ứng với mode lớn nhất không thể được truyền tuyến tính qua nhánh spectral. Pointwise MLP có thể biến đổi và tái phân phối phổ thông qua nonlinear activation, nhưng không làm biến mất giới hạn capacity do spectral bandwidth gây ra. Vì thế số mode là một tham số kiến trúc thực sự, không chỉ là mẹo tăng tốc.

## 7. Vì sao planar FNO chưa đủ cho khí quyển toàn cầu?

Một FFT 2D trên ma trận latitude–longitude ngầm xem hai trục là tuần hoàn độc lập, tức miền có topology của một torus $\mathbb T^2$. Điều đó đúng với longitude nhưng sai với latitude:

- kinh tuyến $0^\circ$ và $360^\circ$ nối nhau;
- Bắc Cực và Nam Cực là hai **điểm**, không phải hai đường biên tuần hoàn;
- khoảng cách vật lý theo longitude co lại theo $\cos\varphi$ khi tiến về cực;
- các ô latitude–longitude có diện tích không đồng đều;
- translation trên ảnh chữ nhật không tương đương rotation trên mặt cầu.

Nếu trực tiếp dùng FFT 2D, model phải học trên một hình học sai: các hàng ở cực bị đối xử như những vòng tròn có chu vi ngang hàng xích đạo, và điều kiện biên latitude trở thành giả tạo. Bonev et al. chỉ ra rằng sự không tương thích này có thể tạo spectral artifacts, làm tiêu tán năng lượng và làm rollout dài hạn kém ổn định.

Vấn đề không phải FFT là một phép biến đổi xấu; vấn đề là **planar Fourier basis không phải eigenbasis tự nhiên của mặt cầu**. Muốn giữ ý tưởng FNO nhưng sửa hình học, ta cần thay nó bằng spherical harmonic basis.

# Phần II — Từ FNO đến Spherical FNO

## 8. Spherical harmonics: Fourier basis của mặt cầu

Dùng colatitude $\theta\in[0,\pi]$ và longitude $\lambda\in[0,2\pi)$, spherical harmonics có dạng

$$
Y_\ell^m(\theta,\lambda)=N_{\ell m}P_\ell^m(\cos\theta)e^{im\lambda},
$$

trong đó $P_\ell^m$ là associated Legendre polynomial, $\ell\ge0$ là degree và $|m|\le\ell$ là order. Chúng tạo cơ sở trực chuẩn của $L^2(\mathbb S^2)$ dưới phần tử diện tích

$$d\Omega=\sin\theta\,d\theta\,d\lambda.$$

Spherical Harmonic Transform (SHT) và biến đổi ngược là

$$
\widehat f_{\ell m}=\int_{0}^{2\pi}\int_{0}^{\pi}
f(\theta,\lambda)\overline{Y_\ell^m(\theta,\lambda)}
\sin\theta\,d\theta\,d\lambda,
$$

$$
f(\theta,\lambda)=\sum_{\ell=0}^{\infty}
\sum_{m=-\ell}^{\ell}\widehat f_{\ell m}Y_\ell^m(\theta,\lambda).
$$

Tương tự Fourier modes trên đường tròn, $\ell$ biểu diễn spatial scale: $\ell$ nhỏ tương ứng với cấu trúc quy mô hành tinh, còn $\ell$ lớn tương ứng với cấu trúc nhỏ hơn. Khác biệt quyết định là basis này phản ánh metric và topology của $\mathbb S^2$.

## 9. Spherical spectral convolution

SFNO giữ nguyên tinh thần của FNO:

$$
\text{spatial field}
\xrightarrow{\mathrm{SHT}}
\text{spherical coefficients}
\xrightarrow{\text{learned mixing}}
\text{new coefficients}
\xrightarrow{\mathrm{SHT}^{-1}}
\text{spatial field}.
$$

Với `operator_type = driscoll-healy`, implementation dùng một ma trận phức $A_\ell$ cho mỗi degree $\ell$ và chia sẻ nó qua mọi order $m$:

$$
\widehat z_{o,\ell m}
=\sum_{c=1}^{E} A_{o,c,\ell}\widehat v_{c,\ell m},
\qquad A_\ell\in\mathbb C^{E\times E}.
$$

Việc trọng số phụ thuộc vào $\ell$ nhưng không phụ thuộc $m$ tạo thành isotropic spherical convolution: cùng một spectral filter được áp cho mọi hướng trong không gian con degree $\ell$. Nó đồng thời giảm số spectral weights từ bậc $O(E^2L^2)$ của kernel phụ thuộc cả $(\ell,m)$ xuống $O(E^2L)$.

Trong code v0.7.4, tensor spectral weight có shape

$$[E_{out},E_{in},L]$$

và dtype `complex64`. Vì vậy một complex tensor element chứa hai real-valued degrees of freedom.

## 10. SHT được tính như thế nào?

`torch-harmonics` triển khai SHT khả vi bằng hai bước chính:

1. FFT theo longitude để chiếu lên $e^{im\lambda}$;
2. quadrature theo latitude để chiếu lên associated Legendre polynomials $P_\ell^m$.

Đây là thuật toán “semi-naive” được thảo luận trong paper SFNO. Toàn bộ phép biến đổi được xây bằng các PyTorch primitives nên gradient có thể truyền qua SHT, spectral multiplication và inverse SHT.

Cần phân biệt hai khái niệm:

- **SHT không phải một lớp học được**: các bảng Legendre/quadrature là cấu trúc toán học cố định;
- **spectral kernel mới là phần học được**: các ma trận phức $A_\ell$ trộn kênh tại từng degree.

Nhờ vậy model dành tham số cho động lực học cần học, còn spherical geometry được encode trực tiếp trong transform. Nền tảng sampling/convolution trên cầu có liên hệ với [Driscoll & Healy, 1994](https://doi.org/10.1006/aama.1994.1008).

# Phần III — Kiến trúc NeuralAtmosphereOperator

## 11. Sơ đồ tổng thể

Với $B$ là batch size, model thực hiện ánh xạ:

$$
\underbrace{[B,26,361,720]}_{\mathbf{x}_t}
\xrightarrow{1\times1\;\text{lifting}}
[B,128,361,720]
\xrightarrow{6\;\text{SFNO blocks}}
[B,128,361,720]
\xrightarrow{1\times1\;\text{projection}}
\underbrace{[B,26,361,720]}_{\Delta\mathbf{x}_t}
\xrightarrow{+\mathbf{x}_t}
\underbrace{[B,26,361,720]}_{\widehat{\mathbf{x}}_{t+\Delta t}}.
$$

Ba tầng chức năng là:

- **Lifting/encoder:** ánh xạ 26 kênh vật lý thành 128 hidden features.
- **SFNO backbone:** thực hiện sáu lần global mixing trên spherical harmonic modes, xen kẽ pointwise MLP và residual connection.
- **Projection/decoder:** ánh xạ hidden features về 26 tendency channels. Wrapper bên ngoài cộng tendency vào trạng thái hiện tại.

Implementation dự án nằm tại [`models/model.py`](../src/neural_atmosphere_operator/models/model.py); cấu hình khóa nằm tại [`configs/model_config.py`](../configs/model_config.py).

## 12. Danh sách và thứ tự 26 input/output channels

Channel dimension không phải 26 features tùy ý mà là một state vector có thứ tự cố định. Sáu surface channels là:

$$
(u_{10m},v_{10m},T_{2m},p_s,p_{msl},TCWV).
$$

Hai mươi pressure-level channels gồm:

- geopotential tại 1000, 850, 500, 250 và 50 hPa;
- zonal wind $u$ tại 1000, 850, 500 và 250 hPa;
- meridional wind $v$ tại 1000, 850, 500 và 250 hPa;
- temperature tại 850, 500, 250 và 100 hPa;
- specific humidity tại 1000 và 850 hPa;
- relative humidity tại 500 hPa.

Encoder đưa cả 26 kênh vào cùng embedding space 128 chiều, nên sau lifting không còn hidden feature nào tương ứng duy nhất với một biến vật lý. Decoder khôi phục đúng thứ tự 26 kênh; phép cộng residual chỉ đúng khi input state và output tendency dùng cùng channel ordering.

SFNO hiện tại xử lý mọi channel như một scalar field trong spectral mixing. Hai thành phần gió $u,v$ là vector components theo local basis, nhưng model không dùng vector spherical harmonics và không áp vector transformation law dưới rotation. Đây là một giới hạn quan trọng: kiến trúc nhận biết spherical geometry tốt hơn planar FFT, nhưng không phải một tensor/vector-field equivariant operator nghiêm ngặt.

## 13. Luồng tensor qua sáu SFNO blocks

`scale_factor = 2` tạo lưới nội bộ

$$
H_i=\frac{H-1}{2}+1=181,
\qquad W_i=\frac{W}{2}=360.
$$

Luồng chính xác là:

| Vị trí | Forward transform | Inverse transform | Output grid |
|---|---|---|---|
| Block 1 | equiangular $361\times720$ | Legendre–Gauss $181\times360$ | internal |
| Block 2–5 | Legendre–Gauss $181\times360$ | Legendre–Gauss $181\times360$ | internal |
| Block 6 | Legendre–Gauss $181\times360$ | equiangular $361\times720$ | full resolution |

Điểm cần nhấn mạnh: đây **không phải** lấy mỗi điểm thứ hai rồi bỏ dữ liệu trước khi model nhìn thấy nó. Block đầu đọc toàn bộ trường $361\times720$, biến đổi nó sang spherical harmonic domain, giữ spectral bandwidth đã cấu hình, rồi tái tạo band-limited representation trên quadrature grid nhỏ hơn. Vì transform đi trước việc đổi grid, đây là spectral restriction/resampling có cấu trúc, không phải spatial decimation tùy tiện.

Legendre–Gauss grid nội bộ phù hợp với quadrature SHT và tiết kiệm activation cho bốn block giữa. Lưới equiangular chỉ được giữ ở biên vào/ra để khớp tensor khí quyển.

## 14. Bên trong một SFNO block của phiên bản 0.7.4

Block topology thực tế không nên bị giản lược thành một FNO block chung chung. Với đầu vào $v_l$, mỗi block thực hiện:

$$
s_l=\operatorname{ISHT}_l
\left(A_l\odot\operatorname{SHT}_l(v_l)\right),
$$

$$
u_l=\operatorname{Norm}_0(s_l),
$$

$$
m_l=W_{2,l}\,\operatorname{GELU}(W_{1,l}u_l+b_{1,l}),
$$

$$
v_{l+1}=r_l+\operatorname{DropPath}\!\left(\operatorname{Norm}_1(m_l)\right).
$$

$r_l$ là residual branch của block input, được spectral-resample khi kích thước vào và ra khác nhau. Cấu hình hiện tại dùng:

- không có inner skip;
- outer skip là identity;
- hai affine InstanceNorm trong mỗi block;
- MLP $128\rightarrow256\rightarrow128$ với GELU;
- `drop_rate = 0` và `drop_path_rate = 0`, nên hai phép dropout hiện là identity.

Outer residual connection duy trì đường truyền thông tin và gradient, đồng thời cho phép block học correction so với representation trước đó. Pointwise MLP thực hiện channel mixing tại mỗi điểm sau global spectral mixing; nhờ nonlinear activation GELU, chuỗi block không suy biến thành một toán tử tuyến tính duy nhất.

## 15. Residual prediction ở cấp trạng thái

Backbone không trực tiếp trả về trạng thái kế tiếp. Nó học tendency/correction trong không gian model:

$$
\Delta\mathbf{x}_t=F_\theta(\mathbf{x}_t),
\qquad
\widehat{\mathbf{x}}_{t+\Delta t}=\mathbf{x}_t+\Delta\mathbf{x}_t.
$$

Lựa chọn này hợp lý vì hai trạng thái cách nhau một bước thời gian thường chia sẻ phần lớn cấu trúc quy mô lớn. Model tập trung capacity vào phần thay đổi thay vì tái tạo lại toàn bộ trạng thái. State-level skip connection cũng tạo một identity mapping tự nhiên: nếu backbone sinh zero tendency, dự báo bằng trạng thái hiện tại.

Dự án cố ý tắt `big_skip` bên trong `torch-harmonics` và chỉ cộng residual một lần trong wrapper. Điều này tránh lỗi **double residual**. Khi đầu vào chứa nhiều trạng thái lịch sử ghép kênh, wrapper lấy 26 kênh cuối làm trạng thái hiện tại; điều kiện cấu hình yêu cầu `in_channels` là bội dương của `out_channels`.

Ánh xạ có thể được lặp tự hồi quy:

$$
\widehat{\mathbf{x}}_{t+(k+1)\Delta t}
=\mathcal G_\theta\!\left(\widehat{\mathbf{x}}_{t+k\Delta t}\right).
$$

Đây là thuộc tính của model và đồ thị tính toán; cách tổ chức huấn luyện hoặc đánh giá rollout nằm ngoài phạm vi notebook.

## 16. Cấu hình model đã chốt

| Tham số | Giá trị | Vai trò |
|---|---:|---|
| `img_size` | `(361, 720)` | Lưới toàn cầu 0.5°, gồm cả hai cực |
| `in_channels` | `26` | Số kênh trạng thái đầu vào khi không ghép history |
| `out_channels` | `26` | Số tendency/state channels đầu ra |
| `scale_factor` | `2` | Lưới nội bộ $181\times360$; ký hiệu SC2 |
| `embed_dim` | `128` | Embedding dimension; ký hiệu E128 |
| `num_layers` | `6` | Sáu SFNO blocks; ký hiệu L6 |
| `operator_type` | `driscoll-healy` | Kernel phụ thuộc $\ell$, chia sẻ qua $m$ |
| `grid` | `equiangular` | Grid geometry của tensor vào/ra |
| `grid_internal` | `legendre-gauss` | Quadrature grid nội bộ |
| `activation_function` | `gelu` | Phi tuyến trong MLP |
| `normalization_layer` | `instance_norm` | Ổn định activation theo từng sample và feature channel |
| `use_mlp` | `True` | Bổ sung pointwise nonlinear channel mixing |
| `mlp_ratio` | `2.0` | MLP hidden dimension $=256$ |
| `hard_thresholding_fraction` | `1.0` | Giữ toàn bộ 180 mode khả dụng sau SC2 |
| `use_complex_kernels` | `True` | Dùng complex-valued spectral weights |
| `pos_embed` | `none` | Không thêm embedding vị trí học được |
| `drop_rate`, `drop_path_rate` | `0.0`, `0.0` | Baseline không stochastic regularization trong backbone |
| `use_residual_connection` | `True` | Dự báo tendency rồi cộng trạng thái hiện tại |

Tên rút gọn đầy đủ là **SFNO-SC2-L6-E128**.

## 17. Số spherical modes thực sự được giữ

Với lưới nội bộ $181\times360$, implementation tính

$$
L_{lat}=181,
\qquad
L_{lon}=\left(\left\lfloor\frac{360}{2}\right\rfloor+1\right)-1=180.
$$

Số retained modes là

$$
L=\left\lfloor
\min(L_{lat},L_{lon})\,f_{HT}
\right\rfloor
=\lfloor180\times1.0\rfloor=180.
$$

`hard_thresholding_fraction = 1.0` nghĩa là không cắt thêm mode nào **bên trong SC2 bandwidth**. Nó không có nghĩa là giữ toàn bộ phổ mà lưới ngoài $361\times720$ có thể biểu diễn. Spectral bottleneck chính đã được xác định bởi internal grid và `scale_factor=2`.

Đây là một trade-off có chủ ý: 180 degree modes vẫn cung cấp global spectral mixing với capacity đáng kể, đồng thời giảm mạnh kích thước activation và transform so với việc giữ full-resolution bandwidth trong cả sáu block.

## 18. Phân bổ và cách tính chính xác số tham số

### 18.1 Spectral kernels

Mỗi block có tensor phức $[128,128,180]$:

$$N_{spec}=6\times128\times128\times180=17{,}694{,}720$$

**complex tensor elements**. Vì mỗi `complex64` gồm phần thực và phần ảo, số real-valued degrees of freedom tương đương là

$$2N_{spec}=35{,}389{,}440.$$

### 18.2 MLP trong sáu block

Mỗi MLP $128\to256\to128$ có bias ở lớp đầu nhưng không có bias ở lớp hai:

$$N_{MLP/block}=128\times256+256+256\times128=65{,}792,$$

$$N_{MLP}=6\times65{,}792=394{,}752.$$

### 18.3 Lifting và projection

Hai convolution $1\times1$ đều không bias:

$$N_{proj}=26\times128+128\times26=6{,}656.$$

### 18.4 InstanceNorm

Mỗi block có hai InstanceNorm; mỗi norm có scale và shift cho 128 kênh:

$$N_{norm}=6\times2\times2\times128=3{,}072.$$

### 18.5 Tổng

PyTorch báo số tensor elements học được là

$$
N_{tensor}=17{,}694{,}720+394{,}752+6{,}656+3{,}072
=\boxed{18{,}099{,}200}.
$$

Nếu đếm từng real-valued scalar độc lập, complex weights phải được nhân đôi:

$$
N_{real\ DOF}=35{,}389{,}440+394{,}752+6{,}656+3{,}072
=\boxed{35{,}793{,}920}.
$$

Hai con số đều đúng nhưng trả lời hai câu hỏi khác nhau. `sum(p.numel())` cho **18.10M tensor elements**; nếu quy đổi complex parameters thành các scalar components thì model có **35.79M real-valued DOF**. Khoảng 98.8% real-valued DOF nằm trong spectral kernels, nên $E$, $L$ và số layer chi phối capacity mạnh hơn số kênh vào/ra.

## 19. Vì sao chọn E128–L6 thay vì cấu hình lớn hơn?

Spectral capacity xấp xỉ

$$N_{spec}\propto N_{layers}E^2L.$$

Do phụ thuộc bậc hai vào embedding dimension, tăng E128 lên E256 không chỉ “gấp đôi model” mà làm riêng số spectral parameters tăng gần bốn lần. E384–L8 còn lớn hơn nhiều và không phù hợp để chọn làm mặc định chỉ vì nó gần một reference configuration.

E128–L6 được chọn vì:

- 26 kênh ít hơn đáng kể so với các model thời tiết lớn hơn;
- độ phân giải 0.5° có bandwidth thấp hơn 0.25°;
- sáu block vẫn cung cấp nhiều vòng global mixing + nonlinear pointwise mixing;
- 180 modes không cắt thêm bởi hard thresholding;
- spectral kernels vẫn mang 35.39M real DOF, nên đây không phải model “siêu nhỏ”;
- cấu hình giữ khoảng trống tài nguyên cho activation toàn cầu, vốn có thể tốn bộ nhớ hơn bản thân weights.

Đây là lựa chọn baseline có kiểm soát, không phải tuyên bố rằng E128 chắc chắn tối ưu. Cấu hình E384–L8 trong repository chỉ là capacity ablation và không được mô tả như bản sao chính thức của Makani.

## 20. Vai trò của các lựa chọn còn lại

### GELU

GELU tạo phi tuyến trơn hơn ReLU và được dùng trong MLP của từng block. Nếu không có phi tuyến, chồng nhiều spectral linear operators và projection tuyến tính cuối cùng vẫn chỉ tạo một ánh xạ tuyến tính, không đủ mô tả động lực học khí quyển.

### InstanceNorm trong backbone

Mỗi InstanceNorm chuẩn hóa activation theo từng sample và từng feature channel trên hai chiều không gian, rồi áp affine scale/shift học được. Nó không lưu running statistics. Thành phần này ổn định activation scale bên trong sáu block; đây là **internal architectural normalization**, khác với tiền xử lý các biến vật lý và được nhắc ở đây chỉ vì nó là một lớp của model.

### Không positional embedding

SHT đã encode spherical geometry và baseline ưu tiên convolution dùng chung trọng số trên toàn cầu. Tắt positional embedding làm giảm tham số phụ thuộc vị trí tuyệt đối và tránh để model ghi nhớ location-specific bias qua một bản đồ học được. Đổi lại, model không nhận thêm tín hiệu vị trí tuyệt đối ngoài những gì có thể suy ra từ state fields và grid.

### Không dropout/drop-path

Baseline giữ backbone deterministic và đơn giản để kiểm chứng. Hai cơ chế vẫn có trong config schema nhưng giá trị zero biến chúng thành identity. Chúng chỉ nên được bật như ablation khi có bằng chứng overfitting.

### Native initialization

Dự án giữ initialization gốc của `torch-harmonics`. Không chạy generic reinitialization lên toàn model vì spectral complex weights, MLP và residual paths sử dụng gain riêng để kiểm soát variance.

# Phần IV — Hàm loss của model

## 21. Vì sao MSE đều trên ma trận là sai hình học?

Trên lưới latitude–longitude, các ô gần cực có diện tích nhỏ hơn các ô gần xích đạo. Nếu tính

$$\frac{1}{BCHW}\sum_{b,c,i,j}(\widehat x_{bcij}-x_{bcij})^2,$$

mỗi grid point được xem như đại diện cho một diện tích bằng nhau, khiến vùng cực nhận trọng số quá lớn so với diện tích thật. Loss chính dùng exact grid-cell area weights.

Gọi biên latitude của hàng $i$ là $\varphi_{i-1/2}$ và $\varphi_{i+1/2}$. Bỏ hằng số longitude chung, diện tích hàng tỉ lệ với

$$
a_i=\left|\sin\varphi_{i+1/2}-\sin\varphi_{i-1/2}\right|.
$$

Sau khi chuẩn hóa về trung bình một,

$$w_i=\frac{a_i}{\frac1H\sum_{r=1}^{H}a_r},$$

loss từng kênh là

$$
\mathcal L_c=\frac{1}{BHW}\sum_{b,i,j}
w_i(\widehat x_{bcij}-x_{bcij})^2.
$$

Dùng latitude-cell boundaries thay vì trực tiếp $\cos\varphi_i$ có một lợi ích nhỏ nhưng quan trọng: hai hàng chứa cực vẫn có diện tích half-cell dương, không bị gán trọng số zero chỉ vì $\cos(\pm90^\circ)=0$.

## 22. Channel-weighted spherical MSE

Các kênh mô tả đại lượng vật lý và mực áp suất khác nhau. Loss tổng hợp là

$$
\mathcal L_{spatial}=\sum_{c=1}^{26}\alpha_c\mathcal L_c.
$$

Baseline dùng quy tắc `auto` tương thích với recipe Makani:

- pressure-level field tại $p$ hPa nhận base weight $0.001p$;
- 2 m temperature nhận $1.0$;
- surface wind và pressure-like field nhận $0.1$;
- kênh không phân loại có fallback $0.01$.

Base weights $q_c$ được chuẩn hóa để $\sum_cq_c=1$. Khi dùng temporal-difference scaling, hệ số cuối là

$$
\alpha_c=q_c\frac{s_c}{s_{\Delta,c}},
$$

với $s_c$ và $s_{\Delta,c}$ là hai statistical scales được cung cấp cho objective. Tỉ lệ này tăng ảnh hưởng của kênh có one-step variation nhỏ so với state scale, tránh để các kênh biến thiên mạnh thống trị gradient. Chi tiết tạo các thống kê hoặc biến đổi dữ liệu không thuộc phạm vi notebook.

Lưu ý implementation chuẩn hóa $q_c$ trước, sau đó nhân tỉ lệ $s_c/s_{\Delta,c}$ và **không chuẩn hóa lại**. Vì vậy cả relative channel weights lẫn absolute loss scale đều có thể thay đổi; đây là hành vi có chủ ý cần được giữ nhất quán khi so sánh checkpoint.

## 23. Loss mặc định và các thành phần tùy chọn

Loss tổng quát trong code là

$$
\mathcal L_{combined}
=\mathcal L_{spatial}+\lambda_{spec}\mathcal L_{spec}.
$$

Baseline khóa $\lambda_{spec}=0$, do đó objective chính chỉ là channel-weighted, spherical-area-weighted MSE. Các thành phần sau tồn tại để nghiên cứu ablation nhưng **đều tắt mặc định**:

### Planar spectral loss

`SpectralLoss` tính sai lệch trên các hệ số `rfft2` của ma trận lat–lon. Nó không phải spherical harmonic loss, không cô lập riêng high frequencies và, với L2, Parseval khiến nó tương đương spatial MSE nếu được chuẩn hóa đúng. Vì model được chọn để khắc phục planar-geometry mismatch, bật planar spectral penalty khi chưa kiểm chứng sẽ làm geometry của objective thiếu nhất quán; do đó mặc định bằng zero.

### Channel-relative loss

Thành phần tùy chọn

$$
\mathcal L_{rel,c}=
\frac{\|\widehat x_c-x_c\|_{2,w}}
{\|x_c\|_{2,w}+\varepsilon}
$$

biểu diễn sai số tương đối so với target norm. Nó hữu ích cho diagnostic/ablation nhưng có thể cạnh tranh với channel weights đã thiết kế, nên weight mặc định bằng zero.

### Backward-only LossScaler

`LossScaler` giữ nguyên forward value nhưng scale gradient của mỗi kênh theo nghịch đảo spatial gradient norm. Đây là cơ chế thực nghiệm kế thừa từ NeuralOceanOperator/MetNet-style balancing. Nó thay đổi optimization dynamics dù scalar loss không đổi và có thể vô hiệu hóa chủ đích của Makani weights; vì vậy mặc định tắt.

### Multi-step loss

API hỗ trợ

$$
\mathcal L_{rollout}=
\frac{\sum_{k=0}^{K-1}\gamma^k\mathcal L_{base}^{(k)}}
{\sum_{k=0}^{K-1}\gamma^k}.
$$

Đây là định nghĩa objective trên nhiều lần áp dụng model, không phải một layer mới trong SFNO. Baseline kiến trúc không phụ thuộc vào việc $K$ bằng bao nhiêu.

## 24. Những gì model cố ý không chứa

Baseline hiện tại không có:

- orography channel;
- land–sea mask;
- solar zenith angle hoặc embedding chu kỳ ngày/năm;
- physics-informed constraint hay conservation penalty;
- post-processing corrector;
- learned positional embedding;
- spherical spectral loss bổ sung.

Đây không phải tuyên bố rằng các thành phần trên vô ích. Đây là quyết định giữ baseline có thể kiểm soát: mọi cải tiến bổ sung đều thay đổi giả thuyết khoa học, nguồn dữ liệu, loss hoặc cách kiểm chứng. Nếu model xuất hiện bias theo địa hình, đất–biển hay chu kỳ bức xạ, các thành phần này là hướng mở rộng hợp lý; nhưng chúng không nên được âm thầm gộp vào baseline rồi khiến nguyên nhân cải thiện/suy giảm không thể truy vết.

Do không có static forcing và astronomical forcing, SFNO hiện tại phải suy ra các dấu hiệu liên quan một cách gián tiếp từ 26 state fields. Vì `pos_embed=none` và Driscoll–Healy kernel chia sẻ trọng số theo rotation, model cũng không có position-specific parameter map học được để ghi nhớ địa hình cố định. Đây là giới hạn cần nêu rõ khi diễn giải kết quả.

## 25. Implementation invariants

Model config và wrapper áp đặt các điều kiện sau:

1. Tensor vào phải đúng shape $[B,C_{in},H,W]$.
2. $(H-1)$ phải chia hết cho `scale_factor` vì lưới latitude gồm cả hai cực.
3. $W$ phải chia hết cho `scale_factor`.
4. Grid ngoài phải là `equiangular`; grid nội bộ chỉ nhận `legendre-gauss` hoặc `equiangular`.
5. Residual prediction yêu cầu $C_{in}\ge C_{out}$ và $C_{in}$ là bội của $C_{out}$.
6. Backbone phải trả đúng $[B,C_{out},H,W]$ trước khi cộng tendency.
7. Chỉ initialization native được chấp nhận để không ghi đè transform-specific scaling.
8. Wrapper kiểm tra API `torch-harmonics` và bắt buộc có cách tắt residual nội bộ.

Các invariants này không chỉ là kiểm tra kiểu dữ liệu. Chúng bảo vệ các giả định hình học và ngăn những lỗi khó thấy như downsampling làm sai cách xử lý hai cực, cộng residual hai lần hoặc load một phiên bản SFNO có block topology khác. Repository vì thế khóa `torch-harmonics==0.7.4`; khi nâng phiên bản cần re-audit topology và numerical behavior.

## 26. Tóm tắt thiết kế

NeuralAtmosphereOperator bắt đầu từ FNO: học global convolution bằng cách biến đổi state fields sang spectral domain, trộn các modes bằng complex weights rồi biến đổi ngược. Planar FNO không mô hình hóa đúng topology và metric của latitude–longitude grid, nên Fourier transform được thay bằng SHT để tạo SFNO.

Baseline **SC2-L6-E128** dùng sáu spherical spectral blocks, 180 modes, Driscoll–Healy kernels, MLP ratio 2, GELU, InstanceNorm và residual prediction. Nó có **18,099,200 PyTorch tensor elements**, tương đương **35,793,920 real-valued trainable scalars** do phần lớn trọng số là complex64.

Objective mặc định là MSE được weighted theo spherical cell area và theo channel/pressure level. Planar spectral loss, relative loss, gradient LossScaler và các physics/corrector components đều không thuộc baseline mặc định. Kết quả là một model có đủ capacity cho thử nghiệm nghiêm túc nhưng vẫn giữ phạm vi khoa học có thể giải thích và kiểm chứng.

## 27. Tài liệu tham khảo

1. Z. Li, N. Kovachki, K. Azizzadenesheli, et al., **Fourier Neural Operator for Parametric Partial Differential Equations**, ICLR 2021. [arXiv:2010.08895](https://arxiv.org/abs/2010.08895).
2. B. Bonev, T. Kurth, C. Hundt, et al., **Spherical Fourier Neural Operators: Learning Stable Dynamics on the Sphere**, ICML 2023, PMLR 202:2806–2823. [PMLR](https://proceedings.mlr.press/v202/bonev23a.html) · [PDF](https://proceedings.mlr.press/v202/bonev23a/bonev23a.pdf).
3. J. R. Driscoll and D. M. Healy, **Computing Fourier Transforms and Convolutions on the 2-Sphere**, Advances in Applied Mathematics 15(2), 1994. [DOI:10.1006/aama.1994.1008](https://doi.org/10.1006/aama.1994.1008).
4. NVIDIA, **torch-harmonics: Differentiable signal processing on the sphere for PyTorch**. [Repository](https://github.com/NVIDIA/torch-harmonics) · [implementation v0.7.4](https://github.com/NVIDIA/torch-harmonics/blob/v0.7.4/torch_harmonics/examples/models/sfno.py).
5. Cấu hình và wrapper của dự án: [`model_config.py`](../configs/model_config.py), [`model.py`](../src/neural_atmosphere_operator/models/model.py), [`loss.py`](../src/neural_atmosphere_operator/models/loss.py).

---

**Quy ước khoa học:** các công thức FNO/SFNO trong notebook mô tả nguyên lý và block topology đúng của phiên bản triển khai đã khóa. Những con số về shape, mode và tham số được suy ra trực tiếp từ config cùng source `torch-harmonics==0.7.4`, không ngoại suy từ một phiên bản thư viện khác.